# Colab #1：`LibriBrain100`（`sub-0` 纯 `sherlock` 114 runs，SimpleB2T `89 train / 8 val / 17 test`）

**推荐双阶段省算力流程**：
1. **阶段 A（CPU High-RAM 实例）**：按 run 流式物化 50 Hz 信号 + `t5-large` 词向量，写完即删临时 250 Hz H5（峰值盘占 <10 GB），打包单文件 `features_libribrain100.tar`（约 7.5 GB）存入 `MyDrive/BrainDecoding_v2/archives/`。
2. **阶段 B（切换至 GPU L4 / A100 实例）**：重新挂载云盘与环境（无需重下 `t5-large`），秒级解包 `.tar` 到 `/content/local_cache`，串行训练并评估第 1 批 **`R1 → R3 → R4`**（`seed=0`）。
3. **阶段 C（确认 `R4 vs R1` 增幅清晰后运行）**：补齐第 2 批 **`R2`（端到端不冻结）** 与 **联合合成平滑脉冲流 `S1–S4`**。

> 如做临时调参试错，请将 `BRAINDECODING_RESULTS_DIR` 改为 `/content/drive/MyDrive/BrainDecoding_v2/scratch_runs`，勿污染正式 `runs/`。

## 阶段 A（CPU 实例）：物化并打包 `.tar` 存云盘
> 若 `MyDrive/BrainDecoding_v2/archives/features_libribrain100.tar` 已存在，直接跳到 **阶段 B**。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%shell
set -e
export PYTHONUNBUFFERED=1
git clone https://github.com/wdongdong0708-jpg/BrainDecoding2.git /content/BrainDecoding || (cd /content/BrainDecoding && git pull)
cd /content/BrainDecoding
python3 -u scripts/colab_setup.py --venv-dir /content/proj-venv --local-cache-dir /content/local_cache

# 仅物化阶段需要拉一次 t5-large（计算出的词向量会一并打进 .tar）
/content/proj-venv/bin/python -u -c "from transformers import AutoTokenizer, AutoModelForTextEncoding; AutoTokenizer.from_pretrained('t5-large'); AutoModelForTextEncoding.from_pretrained('t5-large')"

In [ ]:
%%shell
set -e
export PYTHONUNBUFFERED=1
export HF_HUB_DISABLE_PROGRESS_BARS=1
export HF_HUB_DISABLE_XET=1
export PNPL_OPENNEURO_PROGRESS=0
cd /content/BrainDecoding
export BRAINDECODING_CACHE_DIR=/content/local_cache
export BRAINDECODING_RESULTS_DIR=/content/drive/MyDrive/BrainDecoding_v2/runs
ARCHIVE_DIR=/content/drive/MyDrive/BrainDecoding_v2/archives
PY=/content/proj-venv/bin/python

# 先清理残留 Xet 块缓存与未清理的临时 250 Hz H5（保留已物化的 /content/local_cache 与极小 events.tsv 以便零重下续跑）
rm -rf /root/.cache/huggingface/xet /content/tmp_pnpl_raw/libribrain100/*/derivatives/serialised /content/tmp_pnpl_raw/libribrain100/.cache

echo '=== [1/2] 逐 run 流式物化 LibriBrain100 sub-0 sherlock 114 runs (峰值盘占 <15 GB) ==='
# 逐 run 拉取 1 个 H5 -> 多线程滤波重采样写入 50 Hz .npy -> 立刻删除该 H5（防 200 GB 盘爆满；断线可零重下续跑）
$PY -u -m braindecoding cache \
  --dataset libribrain100 \
  --corpus sherlock \
  --split-policy simpleb2t \
  --pnpl-root /content/tmp_pnpl_raw/libribrain100 \
  --include-test --cleanup-raw --mne-n-jobs 0

echo '=== [2/2] 打包单一 .tar 归档至 Google Drive (BrainDecoding_v2/archives) ==='
# 打包单一 .tar 持久化到 Google Drive（约 7.5 GB）
$PY -u scripts/cache_archive.py pack \
  --modality libribrain100 \
  --cache-dir /content/local_cache \
  --archive-dir "$ARCHIVE_DIR"

ls -lh "$ARCHIVE_DIR/features_libribrain100.tar"

## 阶段 B（切换 GPU L4 / A100 实例后执行）：解包并跑第 1 批 `R1 → R3 → R4`
> 切换运行时后 `/content` 会清空，因此先重新执行下面 3 个轻量准备 Cell（无需再下载 `t5-large`）。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%shell
set -e
export PYTHONUNBUFFERED=1
nvidia-smi
git clone https://github.com/wdongdong0708-jpg/BrainDecoding2.git /content/BrainDecoding || (cd /content/BrainDecoding && git pull)
cd /content/BrainDecoding
python3 -u scripts/colab_setup.py --venv-dir /content/proj-venv --local-cache-dir /content/local_cache

In [ ]:
%%shell
set -e
export PYTHONUNBUFFERED=1
cd /content/BrainDecoding
export BRAINDECODING_CACHE_DIR=/content/local_cache
ARCHIVE_DIR=/content/drive/MyDrive/BrainDecoding_v2/archives
PY=/content/proj-venv/bin/python

$PY -u scripts/cache_archive.py unpack \
  --modality libribrain100 \
  --cache-dir /content/local_cache \
  --archive-dir "$ARCHIVE_DIR" \
  --skip-existing

In [ ]:
%%shell
set -e
export PYTHONUNBUFFERED=1
cd /content/BrainDecoding
export BRAINDECODING_CACHE_DIR=/content/local_cache
export BRAINDECODING_RESULTS_DIR=/content/drive/MyDrive/BrainDecoding_v2/runs
PY=/content/proj-venv/bin/python

# R1: 词编码器（Word-Only）
$PY -u -m braindecoding train \
  --config configs/training/libribrain100_word_encoder.yaml \
  --exp-tag ch2_real_w10 --seed 0 --require-cache --resume auto

# R3: 词编码器冻结 + 整句联合编码（预提取冻结 CNN 特征提速 5-8x）
$PY -u -m braindecoding train \
  --config configs/training/libribrain100_context_transformer.yaml \
  --exp-tag ch2_real_frozen_w10 --word-exp-tag ch2_real_w10 \
  --preextract-frozen-cnn --seed 0 --require-cache --resume auto

# R4: 词编码器冻结 + 整句联合编码 + 式(1)-(2) 错位词间隔窗口去偏
$PY -u -m braindecoding train \
  --config configs/training/siamese_debiased/libribrain100_siamese_debiased_context.yaml \
  --exp-tag ch2_real_debias_w10 --word-exp-tag ch2_real_w10 \
  --seed 0 --require-cache --resume auto

In [ ]:
%%shell
set -e
export PYTHONUNBUFFERED=1
cd /content/BrainDecoding
export BRAINDECODING_CACHE_DIR=/content/local_cache
export BRAINDECODING_RESULTS_DIR=/content/drive/MyDrive/BrainDecoding_v2/runs
PY=/content/proj-venv/bin/python

# 第 1 批留出 test 集统一单次前向评估（eta_eval=0.0 + Pre-Roll 负对照）
$PY -u -m braindecoding eval-run \
  --config configs/training/libribrain100_word_encoder.yaml \
  --exp-tag ch2_real_w10 --seed 0 --split test --require-cache --no-plot

$PY -u -m braindecoding eval-run \
  --config configs/training/libribrain100_context_transformer.yaml \
  --exp-tag ch2_real_frozen_w10 --word-exp-tag ch2_real_w10 --seed 0 --split test \
  --control-config configs/controls/temporal_shift/temporal_shift_control.yaml \
  --shift-seconds -10,-20,-30,-40 --shift-mode preroll --require-cache --no-plot

$PY -u -m braindecoding eval-run \
  --config configs/training/siamese_debiased/libribrain100_siamese_debiased_context.yaml \
  --exp-tag ch2_real_debias_w10 --word-exp-tag ch2_real_w10 --seed 0 --split test \
  --control-config configs/controls/temporal_shift/temporal_shift_control.yaml \
  --shift-seconds -10,-20,-30,-40 --shift-mode preroll --require-cache --no-plot

## 阶段 C（可选，第 1 批确认趋势后执行）：第 2 批 `R2` + 联合合成信号 `S1–S4`

In [ ]:
%%shell
set -e
export PYTHONUNBUFFERED=1
cd /content/BrainDecoding
export BRAINDECODING_CACHE_DIR=/content/local_cache
export BRAINDECODING_RESULTS_DIR=/content/drive/MyDrive/BrainDecoding_v2/runs
PY=/content/proj-venv/bin/python

# R2: 随机初始化端到端不冻结 CNN + 16 层 Transformer
$PY -u -m braindecoding train \
  --config configs/training/libribrain100_context_transformer.yaml \
  --exp-tag ch2_real_unfrozen_w10 --from-scratch --seed 0 --require-cache --resume auto

$PY -u -m braindecoding eval-run \
  --config configs/training/libribrain100_context_transformer.yaml \
  --exp-tag ch2_real_unfrozen_w10 --seed 0 --split test --require-cache --no-plot

In [ ]:
%%shell
set -e
export PYTHONUNBUFFERED=1
cd /content/BrainDecoding
export BRAINDECODING_CACHE_DIR=/content/local_cache
export BRAINDECODING_RESULTS_DIR=/content/drive/MyDrive/BrainDecoding_v2/runs
PY=/content/proj-venv/bin/python

# S1 -> S2 -> S3 -> S4: 联合合成平滑脉冲流（零读取真实 MEG）
$PY -u -m braindecoding train \
  --config configs/training/libribrain100_word_encoder.yaml \
  --exp-tag ch2_synth_w10 --synthetic-timing shared --seed 0 --require-cache --resume auto

$PY -u -m braindecoding train \
  --config configs/training/libribrain100_context_transformer.yaml \
  --exp-tag ch2_synth_unfrozen_w10 --from-scratch --synthetic-timing shared --seed 0 --require-cache --resume auto

$PY -u -m braindecoding train \
  --config configs/training/libribrain100_context_transformer.yaml \
  --exp-tag ch2_synth_frozen_w10 --word-exp-tag ch2_synth_w10 --synthetic-timing shared --seed 0 --require-cache --resume auto

$PY -u -m braindecoding train \
  --config configs/training/siamese_debiased/libribrain100_siamese_debiased_context.yaml \
  --exp-tag ch2_synth_debias_w10 --word-exp-tag ch2_synth_w10 --synthetic-timing shared --seed 0 --require-cache --resume auto

# 留出 test 集评估（自动从检查点继承 synthetic_timing=shared）
$PY -u -m braindecoding eval-run --config configs/training/libribrain100_word_encoder.yaml --exp-tag ch2_synth_w10 --seed 0 --split test --require-cache --no-plot
$PY -u -m braindecoding eval-run --config configs/training/libribrain100_context_transformer.yaml --exp-tag ch2_synth_unfrozen_w10 --seed 0 --split test --require-cache --no-plot
$PY -u -m braindecoding eval-run --config configs/training/libribrain100_context_transformer.yaml --exp-tag ch2_synth_frozen_w10 --word-exp-tag ch2_synth_w10 --seed 0 --split test --require-cache --no-plot
$PY -u -m braindecoding eval-run --config configs/training/siamese_debiased/libribrain100_siamese_debiased_context.yaml --exp-tag ch2_synth_debias_w10 --word-exp-tag ch2_synth_w10 --seed 0 --split test --require-cache --no-plot

## 阶段 D（方案体系 II 最小改动 `seed=0` 实测 + 新旧对比图 2a/2b/9 体检）
> 复用已解包的 `/content/local_cache`，隔离输出至 `scratch_runs/`（不污染正式 `runs/`），**保留 1.0s 词窗两阶段与 $r \in \{0,1,2\}$ 框架，不额外压制真实听觉韵律**：
> 1. **洗净 Stage-1（`s2_word_causal_hp_mingap`）**：修掉批内 `99.78%` 重叠负样本（`MinGapBatchSampler`, $\Delta_{\min}=2.5\text{s}$）与词后 `[0, 0.5s]` 自减基线（改用 `2.0s` 因果滑动均值高通 + `baseline_seconds=0.0`），`BrainModule` 参数结构零改动；
> 2. **第 2 章整句去偏（`s2_ch2_salience_add`）**：冻结新 Stage-1 + 词内显著性门控加性跨词注意力（`Wiener-AG2`）+ 超额相干错位零空间；
> 3. **第 3 章局部前瞻（`s2_ch3_salience_r2`）**：同底座 + 局部前缀编码（`K0 / K1 / K2`, $r \in \{0,1,2\}$）；
> 4. **真底座联合合成流负对照（`s2_ch2_synth_real_cnn`）**：冻结新 Stage-1 跑联合合成脉冲流，验证零切分泄露（`~20%` 随机线）。

In [ ]:
%%shell
set -e
export PYTHONUNBUFFERED=1
cd /content/BrainDecoding && git pull
export BRAINDECODING_CACHE_DIR=/content/local_cache
export BRAINDECODING_RESULTS_DIR=/content/drive/MyDrive/BrainDecoding_v2/scratch_runs
PY=/content/proj-venv/bin/python

# 1) 重训洗净后的 Stage-1 Word-Only（2.0s 因果滑动高通 + baseline=0.0 + 2.5s MinGapBatchSampler）
$PY -u -m braindecoding train \
  --config configs/training/libribrain100_word_encoder_causal_highpass_mingap.yaml \
  --exp-tag s2_word_causal_hp_mingap \
  --seed 0 --require-cache --resume auto --no-progress

WORD_CKPT="/content/drive/MyDrive/BrainDecoding_v2/scratch_runs/libribrain100/s2_word_causal_hp_mingap/word/all-n01/seed-000/best.ckpt"
echo "Using cleaned Stage-1 checkpoint: $WORD_CKPT"

# 2) 阶段 2.1（第 2 章整句）：冻结新 Stage-1 + 词内显著性门控加性跨词注意力 + ExcessNull
$PY -u -m braindecoding train \
  --config configs/training/siamese_debiased/libribrain100_causal_highpass_salience_additive_context.yaml \
  --exp-tag s2_ch2_salience_add --word-checkpoint "$WORD_CKPT" \
  --seed 0 --require-cache --resume auto --no-progress

# 3) 阶段 2.3（第 3 章局部前瞻 r=0,1,2）：冻结新 Stage-1 + 词内显著性门控加性局部前缀编码 + ExcessNull
$PY -u -m braindecoding train \
  --config configs/training/siamese_debiased/libribrain100_causal_highpass_salience_additive_prefix_revision.yaml \
  --exp-tag s2_ch3_salience_r2 --word-checkpoint "$WORD_CKPT" \
  --seed 0 --require-cache --resume auto --no-progress

# 4) 冻结新 Stage-1 CNN 的联合合成脉冲流负对照（验证纯噪声回到 ~20% 随机水平）
$PY -u -m braindecoding train \
  --config configs/training/siamese_debiased/libribrain100_causal_highpass_salience_additive_context.yaml \
  --exp-tag s2_ch2_synth_real_cnn --word-checkpoint "$WORD_CKPT" \
  --synthetic-timing shared --epochs 12 --seed 0 --require-cache --resume auto --no-progress

# 留出 test 集标准评估（自动从检查点继承 2.0s 因果高通 + baseline=0.0 预处理）
$PY -u -m braindecoding eval-run \
  --config configs/training/libribrain100_word_encoder_causal_highpass_mingap.yaml \
  --exp-tag s2_word_causal_hp_mingap --seed 0 --split test \
  --control-config configs/controls/temporal_shift/temporal_shift_control.yaml \
  --shift-seconds -10,-20,-30,-40 --shift-mode preroll --require-cache --no-plot --no-progress

for TAG_CFG in \
  "s2_ch2_salience_add:configs/training/siamese_debiased/libribrain100_causal_highpass_salience_additive_context.yaml" \
  "s2_ch3_salience_r2:configs/training/siamese_debiased/libribrain100_causal_highpass_salience_additive_prefix_revision.yaml"; do
  TAG="${TAG_CFG%%:*}"
  CFG="${TAG_CFG#*:}"
  $PY -u -m braindecoding eval-run \
    --config "$CFG" --exp-tag "$TAG" --seed 0 --split test \
    --control-config configs/controls/temporal_shift/temporal_shift_control.yaml \
    --shift-seconds -10,-20,-30,-40 --shift-mode preroll --require-cache --no-plot --no-progress
done

$PY -u -m braindecoding eval-run \
  --config configs/training/siamese_debiased/libribrain100_causal_highpass_salience_additive_context.yaml \
  --exp-tag s2_ch2_synth_real_cnn --seed 0 --split test --require-cache --no-plot --no-progress

In [ ]:
%%shell
set -e
export PYTHONUNBUFFERED=1
cd /content/BrainDecoding
export BRAINDECODING_CACHE_DIR=/content/local_cache
export BRAINDECODING_RESULTS_DIR=/content/drive/MyDrive/BrainDecoding_v2/scratch_runs
PY=/content/proj-venv/bin/python

if [ -f /content/drive/MyDrive/BrainDecoding_v2/runs/libribrain100/ch2_real_w10/word/all-n01/seed-000/best.ckpt ]; then
  MAIN_RUNS=/content/drive/MyDrive/BrainDecoding_v2/runs/libribrain100
else
  MAIN_RUNS=/content/drive/MyDrive/libribrain100
fi
AB_RUNS=/content/drive/MyDrive/BrainDecoding_v2/scratch_runs/libribrain100
OUT_JSON=/content/drive/MyDrive/BrainDecoding_v2/exports/libribrain100_scheme2_pilot_diag_seed0.json
mkdir -p /content/drive/MyDrive/BrainDecoding_v2/exports

# 同表横向对比旧基线（自动按各自检查点的基线/高通预处理独立评测）vs 方案体系 II 新模型
MODEL_ARGS=()
if [ -f "$MAIN_RUNS/ch2_real_w10/word/all-n01/seed-000/best.ckpt" ]; then
  MODEL_ARGS+=( --model "Old_WordOnly=$MAIN_RUNS/ch2_real_w10/word/all-n01/seed-000/best.ckpt" )
fi
if [ -f "$MAIN_RUNS/ch2_real_debias_w10_cm5_a05_lam10/context/all-n01/seed-000/best.ckpt" ]; then
  MODEL_ARGS+=( --model "Old_Debias=$MAIN_RUNS/ch2_real_debias_w10_cm5_a05_lam10/context/all-n01/seed-000/best.ckpt" )
elif [ -f "$MAIN_RUNS/ch2_real_debias_w10/context/all-n01/seed-000/best.ckpt" ]; then
  MODEL_ARGS+=( --model "Old_Debias=$MAIN_RUNS/ch2_real_debias_w10/context/all-n01/seed-000/best.ckpt" )
fi
MODEL_ARGS+=(
  --model "S2_WordOnly=$AB_RUNS/s2_word_causal_hp_mingap/word/all-n01/seed-000/best.ckpt"
  --model "S2_Ch2_SalienceAdd=$AB_RUNS/s2_ch2_salience_add/context/all-n01/seed-000/best.ckpt"
  --model "S2_Ch3_Salience=$AB_RUNS/s2_ch3_salience_r2/prefix_revision/all-n01/seed-000/best.ckpt"
)

# 运行完整 Clean 图 2a / 图 2b（含 prev_ioi）+ 图 9 跨录音零重叠拼接体检
$PY -u scripts/diagnose_duration_shortcut.py \
  --dataset libribrain100 \
  --split test \
  --seed 0 \
  --eval-batch-size 512 \
  --no-progress \
  "${MODEL_ARGS[@]}" \
  --output-json "$OUT_JSON"